# BUS-BRA: Gemma 3 + MedGemma — real ultrasound vs white or black blank-image control (paired analysis)

**No GPU inference.** This notebook compares the results you already obtained for Gemma 3 4B and MedGemma 4B on the original frozen 400 samples against a **derived blank-image baseline** obtained from five verified same-input inferences per model/color.

1. Run the original Gemma/MedGemma notebook and the separate blank-control notebook.
2. Place the original `vqa_results_gemma3_4b.csv` and `vqa_results_medgemma_4b.csv` into `./real_results/` (or change the path below).
3. Place the entire white/black control output directory at `./blank_controls/` (or change the path below).
4. Select `CONTROL_COLOR = 'white'` or `'black'` in the configuration cell and run all cells. For the second color, change the setting and run all cells again. Outputs are stored in **different color-named folders**.

**Interpretation:** The blank-control 400-row CSVs are repeated predictions *derived from five identical-input stability checks*, not 400 independent blank-image classifications. The paired comparison tests whether the real ultrasound changes predictions and correctness, but not whether the VLM has learned clinically meaningful imaging features.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Set paths for the ORIGINAL results and the dual-color CONTROL root.
REAL_RESULTS_DIR = Path('./real_results')
BLANK_RESULTS_BASE = Path('./blank_controls')
CONTROL_COLOR = 'white'  # Change to 'black' and Run All again for the black control.
if CONTROL_COLOR not in {'white', 'black'}:
    raise ValueError('CONTROL_COLOR must be white or black')
BLANK_RESULTS_DIR = BLANK_RESULTS_BASE / CONTROL_COLOR
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists()
              else Path('.')) / f'bus_bra_paired_comparison_gemma_medgemma_{CONTROL_COLOR}'

EXPECTED_N = 400
EXPECTED_CLASS_COUNTS = {'benign': 200, 'malignant': 200}
MODEL_TAGS = {
    'Gemma 3 4B': 'gemma3_4b',
    'MedGemma 4B': 'medgemma_4b',
}
BOOTSTRAP_REPS = 10_000
BOOTSTRAP_SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Real input:', REAL_RESULTS_DIR.resolve())
print('Control:', CONTROL_COLOR, '| control input:', BLANK_RESULTS_DIR.resolve())
print('Output:', OUTPUT_DIR.resolve())


## 1. Load and strictly pair the predictions

`question_id` identifies the frozen ultrasound case. We verify that both runs have exactly the same 400 IDs, 200 benign and 200 malignant reference labels, and the same question text. The labels are used **only to score predictions**, not as model inputs.

In [ ]:
REQUIRED_COLUMNS = {'question_id', 'true_answer', 'predicted_answer', 'question', 'raw_output'}
CLASSES = {'benign', 'malignant'}
ALLOWED_PREDICTIONS = CLASSES | {'unparsed'}


def load_result(path, condition):
    if not path.is_file():
        raise FileNotFoundError(
            f'{condition}: Missing {path}. Check your directory configuration and the downloaded result files.'
        )
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'{condition}: Missing expected columns: {sorted(missing)}')
    if len(df) != EXPECTED_N:
        raise ValueError(f'{condition}: Expected {EXPECTED_N} rows, got {len(df)}. '
                         'Finish the inference before comparing.')
    if df.question_id.eq('').any() or df.question_id.duplicated().any():
        raise ValueError(f'{condition}: Missing/duplicate question_id values.')
    for column in ('true_answer', 'predicted_answer'):
        df[column] = df[column].str.strip().str.lower()
    if not set(df.true_answer).issubset(CLASSES):
        raise ValueError(f'{condition}: Invalid ground-truth labels: {set(df.true_answer) - CLASSES}')
    if df.true_answer.value_counts().to_dict() != EXPECTED_CLASS_COUNTS:
        raise ValueError(f'{condition}: Wrong reference-label distribution: '
                         f'{df.true_answer.value_counts().to_dict()}')
    invalid_predictions = set(df.predicted_answer) - ALLOWED_PREDICTIONS
    if invalid_predictions:
        raise ValueError(f'{condition}: Invalid predictions: {invalid_predictions}')
    return df


def pair_results(model_name, tag):
    filename = f'vqa_results_{tag}.csv'
    real = load_result(REAL_RESULTS_DIR / filename, f'{model_name} real')
    blank = load_result(BLANK_RESULTS_DIR / filename, f'{model_name} blank')
    real_ids, blank_ids = set(real.question_id), set(blank.question_id)
    if real_ids != blank_ids:
        raise ValueError(f'{model_name}: Different sample IDs. '
                         f'Missing from blank={sorted(real_ids - blank_ids)[:5]}; '
                         f'extra in blank={sorted(blank_ids - real_ids)[:5]}')

    merged = real.merge(blank, on='question_id', how='inner', suffixes=('_real', '_blank'),
                        validate='one_to_one')
    for col in ('true_answer', 'question'):
        left, right = f'{col}_real', f'{col}_blank'
        if not merged[left].equals(merged[right]):
            raise ValueError(f'{model_name}: Mismatched {col} for the same question_id.')
    if 'group_real' in merged and 'group_blank' in merged:
        if not merged['group_real'].equals(merged['group_blank']):
            raise ValueError(f'{model_name}: Mismatched groups for the same question_id.')

    merged = merged.rename(columns={
        'true_answer_real': 'true_answer',
        'question_real': 'question',
        'predicted_answer_real': 'real_prediction',
        'predicted_answer_blank': 'blank_prediction',
        'raw_output_real': 'real_raw_output',
        'raw_output_blank': 'blank_raw_output',
    })
    merged = merged.drop(columns=['true_answer_blank', 'question_blank'])
    merged['real_correct'] = merged.real_prediction.eq(merged.true_answer)
    merged['blank_correct'] = merged.blank_prediction.eq(merged.true_answer)
    merged['prediction_changed'] = merged.real_prediction.ne(merged.blank_prediction)
    conditions = [
        merged.real_correct & ~merged.blank_correct,
        ~merged.real_correct & merged.blank_correct,
        merged.real_correct & merged.blank_correct,
    ]
    choices = ['real_only_correct', 'blank_only_correct', 'both_correct']
    merged['paired_outcome'] = np.select(conditions, choices, default='both_wrong')
    merged.insert(0, 'model', model_name)
    return merged.sort_values('question_id').reset_index(drop=True)

PAIRED = {name: pair_results(name, tag) for name, tag in MODEL_TAGS.items()}
for model, table in PAIRED.items():
    print(f'{model}: matched {len(table)}/{EXPECTED_N} cases; '
          f'{table.prediction_changed.sum()} different predictions')


## 2. Paired metrics and uncertainty

The accuracy improvement (real minus blank) is exactly:

\[\Delta \text{accuracy} = \frac{\#(\text{real only correct}) - \#(\text{blank only correct})}{400}\]

We also report an **exact two-sided McNemar p-value** (via a binomial test on discordant correctness pairs) and a **paired bootstrap 95% confidence interval** for the accuracy difference. Both tests use case-level pairing. As this dataset has one image per Case, the pairing avoids double-counting multiple images from a single Case.

In [ ]:
from scipy.stats import binomtest


def summarize_paired(df, *, bootstrap_reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    n = len(df)
    counts = df.paired_outcome.value_counts()
    real_only = int(counts.get('real_only_correct', 0))
    blank_only = int(counts.get('blank_only_correct', 0))
    both_correct = int(counts.get('both_correct', 0))
    both_wrong = int(counts.get('both_wrong', 0))
    discordant = real_only + blank_only
    p_value = (float(binomtest(min(real_only, blank_only), discordant, p=0.5).pvalue)
               if discordant else 1.0)

    # Each item is one matched Case: +100 if real correct / blank wrong,
    # -100 for the opposite, and 0 when correctness is tied.
    differences = (df.real_correct.astype(int) - df.blank_correct.astype(int)).to_numpy()
    rng = np.random.default_rng(seed)
    boot_means = np.empty(bootstrap_reps, dtype=float)
    for b in range(bootstrap_reps):
        boot_means[b] = differences[rng.integers(0, n, n)].mean() * 100
    low, high = np.quantile(boot_means, [0.025, 0.975])

    return {
        'model': df.model.iat[0],
        'n': n,
        'real_correct': int(df.real_correct.sum()),
        'blank_correct': int(df.blank_correct.sum()),
        'real_accuracy_pct': round(float(df.real_correct.mean() * 100), 3),
        'blank_accuracy_pct': round(float(df.blank_correct.mean() * 100), 3),
        'real_minus_blank_pp': round(float(differences.mean() * 100), 3),
        'paired_bootstrap_95ci_low_pp': round(float(low), 3),
        'paired_bootstrap_95ci_high_pp': round(float(high), 3),
        'real_only_correct': real_only,
        'blank_only_correct': blank_only,
        'both_correct': both_correct,
        'both_wrong': both_wrong,
        'changed_predictions': int(df.prediction_changed.sum()),
        'changed_predictions_pct': round(float(df.prediction_changed.mean() * 100), 3),
        'mcnemar_exact_two_sided_p': p_value,
        'blank_predicts_benign': int(df.blank_prediction.eq('benign').sum()),
        'blank_predicts_malignant': int(df.blank_prediction.eq('malignant').sum()),
        'blank_unparsed': int(df.blank_prediction.eq('unparsed').sum()),
        'real_unparsed': int(df.real_prediction.eq('unparsed').sum()),
    }

summary = pd.DataFrame([summarize_paired(df) for df in PAIRED.values()])
display(summary)

per_class_rows = []
for name, df in PAIRED.items():
    for label, group in df.groupby('true_answer'):
        per_class_rows.append({
            'model': name,
            'true_answer': label,
            'n': len(group),
            'real_accuracy_pct': round(group.real_correct.mean() * 100, 3),
            'blank_accuracy_pct': round(group.blank_correct.mean() * 100, 3),
            'real_only_correct': int((group.real_correct & ~group.blank_correct).sum()),
            'blank_only_correct': int((~group.real_correct & group.blank_correct).sum()),
        })
per_class = pd.DataFrame(per_class_rows)
display(per_class)


## 3. Examine which answers changed

The outputs below reveal whether real-image input actually fixes incorrect blank-image answers. They also expose cases where blank-image predictions were correct but real-image predictions became wrong. Examine `real_raw_output` and `blank_raw_output` for parser/truncation issues.

In [ ]:
COMBINED = pd.concat(PAIRED.values(), ignore_index=True)

# Check the paired categories for each model.
display(pd.crosstab(COMBINED.model, COMBINED.paired_outcome))

# Show cases where real ultrasound fixed an error from the blank-image control.
real_helps = COMBINED.loc[COMBINED.paired_outcome.eq('real_only_correct'),
    ['model', 'question_id', 'true_answer', 'real_prediction', 'blank_prediction',
     'real_raw_output', 'blank_raw_output']]
print('Example cases helped by real images:')
display(real_helps.head(15))

# Show cases where the real image made the prediction worse.
real_hurts = COMBINED.loc[COMBINED.paired_outcome.eq('blank_only_correct'),
    ['model', 'question_id', 'true_answer', 'real_prediction', 'blank_prediction',
     'real_raw_output', 'blank_raw_output']]
print('Example cases hurt by real images:')
display(real_hurts.head(15))


## 4. Visualize and export results

Save tables for a paper/report and a compact accuracy chart. The file `paired_predictions_all_models.csv` contains every aligned prediction and its correctness category.

In [ ]:
import matplotlib.pyplot as plt

summary.to_csv(OUTPUT_DIR / 'paired_model_summary.csv', index=False)
per_class.to_csv(OUTPUT_DIR / 'paired_per_class_summary.csv', index=False)
COMBINED.to_csv(OUTPUT_DIR / 'paired_predictions_all_models.csv', index=False)
for model, tag in MODEL_TAGS.items():
    PAIRED[model].to_csv(OUTPUT_DIR / f'paired_predictions_{tag}.csv', index=False)

# Matplotlib default palette; user can restyle later for publication.
plot_df = summary.set_index('model')[['real_accuracy_pct', 'blank_accuracy_pct']]
ax = plot_df.plot(kind='bar', figsize=(9, 5), rot=0)
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('')
ax.set_ylim(0, 100)
ax.set_title(f'BUS-BRA: real ultrasound vs {CONTROL_COLOR} blank image (400 cases)')
ax.legend(['Real ultrasound', f'{CONTROL_COLOR.capitalize()} blank control'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'real_vs_{CONTROL_COLOR}_accuracy.png', dpi=200)
plt.show()

report_lines = [
    f'# BUS-BRA paired real-image vs {CONTROL_COLOR} blank-image control',
    '',
    'All comparisons pair 400 identical BUS-BRA Case IDs, with 200 benign and 200 malignant cases.',
    'Unparsed labels are scored as wrong. Percentage-point improvement is real minus blank accuracy.',
    'Exact McNemar p-values test discordant paired correctness; bootstrap 95% CIs resample paired cases.',
    '',
]
for _, r in summary.iterrows():
    report_lines += [
        f'## {r["model"]}',
        f'- Real-image accuracy: {r["real_accuracy_pct"]:.2f}% ({int(r["real_correct"])}/400)',
        f'- Blank-image accuracy: {r["blank_accuracy_pct"]:.2f}% ({int(r["blank_correct"])}/400)',
        f'- Accuracy difference: {r["real_minus_blank_pp"]:+.2f} percentage points '
          f'(paired bootstrap 95% CI {r["paired_bootstrap_95ci_low_pp"]:+.2f} to '
          f'{r["paired_bootstrap_95ci_high_pp"]:+.2f})',
        f'- Cases helped by real image: {int(r["real_only_correct"])}; '
          f'hurt by real image: {int(r["blank_only_correct"])}',
        f'- Changed predictions: {int(r["changed_predictions"])}/400 '
          f'({r["changed_predictions_pct"]:.2f}%)',
        f'- Exact McNemar p-value: {r["mcnemar_exact_two_sided_p"]:.5g}',
        f'- Blank predicted benign/malignant/unparsed: '
          f'{int(r["blank_predicts_benign"])}/'
          f'{int(r["blank_predicts_malignant"])}/'
          f'{int(r["blank_unparsed"])}',
        '',
    ]
(OUTPUT_DIR / 'comparison_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

print('Saved comparison files to:', OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)

# A consistent blank-only classification is expected for this fixed prompt / white image setup.
if any((summary['blank_predicts_benign'] == EXPECTED_N) |
       (summary['blank_predicts_malignant'] == EXPECTED_N)):
    print('\nNOTE: A model predicted one class for all blank images. With 200 cases per class, '
          'its blank accuracy is exactly 50%. This is a useful baseline, not proof of visual reasoning.')
